In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
get_ipython().system('pip install rasterio --quiet')
get_ipython().system('pip install geopandas --quiet')
print("Dependencies installed.")

Dependencies installed.


In [3]:
# ============================================================
# CELL 2 — Download Sen1Floods11 dataset (Sentinel-2 subset)
# ============================================================
import subprocess
import os
 
os.makedirs("/kaggle/working/data/S2Hand", exist_ok=True)
os.makedirs("/kaggle/working/data/LabelHand", exist_ok=True)
 
print("Downloading Sentinel-2 flood image chips...")
result = subprocess.run([
    "gsutil", "-m", "cp", "-r",
    "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/",
    "/kaggle/working/data/"
], capture_output=True, text=True)
print("S2 images:", result.returncode)
 
result2 = subprocess.run([
    "gsutil", "-m", "cp", "-r",
    "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/",
    "/kaggle/working/data/"
], capture_output=True, text=True)
print("Labels:", result2.returncode)
print("Download complete.")

S2 images: 0
Labels: 0
Download complete.


In [4]:
# ============================================================
# CELL 3 — Verify download and count files
# ============================================================
s2_dir = "/kaggle/working/data/S2Hand"
label_dir = "/kaggle/working/data/LabelHand"
 
s2_files = sorted([f for f in os.listdir(s2_dir) if f.endswith(".tif")])
label_files = sorted([f for f in os.listdir(label_dir) if f.endswith(".tif")])
 
print(f"Sentinel-2 image chips found: {len(s2_files)}")
print(f"Label mask files found:       {len(label_files)}")

Sentinel-2 image chips found: 446
Label mask files found:       446


In [5]:
# ============================================================
# CELL 4 — Compute NDWI baseline (naive threshold 0.3) across all chips
# ============================================================
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, jaccard_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')
 
NDWI_THRESHOLD = 0.3
results = []
 
for s2_name in s2_files:
    base_name = s2_name.replace("_S2Hand.tif", "")
    label_name = base_name + "_LabelHand.tif"
    label_path = os.path.join(label_dir, label_name)
    if not os.path.exists(label_path):
        continue
 
    with rasterio.open(os.path.join(s2_dir, s2_name)) as src:
        data = src.read().astype(np.float32)
    with rasterio.open(label_path) as lsrc:
        label = lsrc.read(1)
 
    green = data[2]
    nir = data[7]
    denom = green + nir
    denom[denom == 0] = 1e-6
    ndwi = (green - nir) / denom
    pred = (ndwi > NDWI_THRESHOLD).astype(np.int8)
 
    valid_mask = (label != -1)
    y_true = label[valid_mask].astype(np.int8)
    y_pred = pred[valid_mask]
    if len(y_true) == 0 or y_true.sum() == 0:
        continue
 
    results.append({
        "chip": base_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_flood": f1_score(y_true, y_pred, zero_division=0),
        "iou_flood": jaccard_score(y_true, y_pred, zero_division=0),
    })
 
accs = [r["accuracy"] for r in results]
f1s = [r["f1_flood"] for r in results]
ious = [r["iou_flood"] for r in results]
print(f"Naive NDWI (threshold=0.3): Mean F1={np.mean(f1s):.4f}, Mean IoU={np.mean(ious):.4f}")

Naive NDWI (threshold=0.3): Mean F1=0.0859, Mean IoU=0.0637


In [6]:
# ============================================================
# CELL 5 — Sweep NDWI thresholds to find best baseline
# ============================================================
thresholds_to_test = np.arange(-0.3, 0.6, 0.05)
threshold_results = []
 
for thresh in thresholds_to_test:
    all_f1, all_iou = [], []
    for s2_name in s2_files:
        base_name = s2_name.replace("_S2Hand.tif", "")
        label_path = os.path.join(label_dir, base_name + "_LabelHand.tif")
        if not os.path.exists(label_path):
            continue
        with rasterio.open(os.path.join(s2_dir, s2_name)) as src:
            data = src.read().astype(np.float32)
        with rasterio.open(label_path) as lsrc:
            label = lsrc.read(1)
        green, nir = data[2], data[7]
        denom = green + nir
        denom[denom == 0] = 1e-6
        ndwi = (green - nir) / denom
        pred = (ndwi > thresh).astype(np.int8)
        valid_mask = (label != -1)
        y_true = label[valid_mask].astype(np.int8)
        y_pred = pred[valid_mask]
        if len(y_true) == 0 or y_true.sum() == 0:
            continue
        all_f1.append(f1_score(y_true, y_pred, zero_division=0))
        all_iou.append(jaccard_score(y_true, y_pred, zero_division=0))
    threshold_results.append({"threshold": thresh, "f1": np.mean(all_f1), "iou": np.mean(all_iou)})
    print(f"Threshold {thresh:+.2f}  F1: {np.mean(all_f1):.4f}  IoU: {np.mean(all_iou):.4f}")
 
best = max(threshold_results, key=lambda r: r["f1"])
print(f"\nBEST NDWI THRESHOLD: {best['threshold']:+.2f}  F1={best['f1']:.4f}  IoU={best['iou']:.4f}")
NDWI_BEST_THRESHOLD = best['threshold']  # expect ~ -0.10
 

Threshold -0.30  F1: 0.3406  IoU: 0.2505
Threshold -0.25  F1: 0.4243  IoU: 0.3209
Threshold -0.20  F1: 0.5283  IoU: 0.4166
Threshold -0.15  F1: 0.6070  IoU: 0.4976
Threshold -0.10  F1: 0.6289  IoU: 0.5252
Threshold -0.05  F1: 0.5920  IoU: 0.4922
Threshold -0.00  F1: 0.5219  IoU: 0.4275
Threshold +0.05  F1: 0.4311  IoU: 0.3447
Threshold +0.10  F1: 0.3353  IoU: 0.2641
Threshold +0.15  F1: 0.2499  IoU: 0.1931
Threshold +0.20  F1: 0.1753  IoU: 0.1329
Threshold +0.25  F1: 0.1231  IoU: 0.0929
Threshold +0.30  F1: 0.0860  IoU: 0.0637
Threshold +0.35  F1: 0.0568  IoU: 0.0410
Threshold +0.40  F1: 0.0308  IoU: 0.0218
Threshold +0.45  F1: 0.0160  IoU: 0.0122
Threshold +0.50  F1: 0.0093  IoU: 0.0074
Threshold +0.55  F1: 0.0028  IoU: 0.0020

BEST NDWI THRESHOLD: -0.10  F1=0.6289  IoU=0.5252


In [7]:
# ============================================================
# CELL 6 — PyTorch setup
# ============================================================
get_ipython().system('pip install torch torchvision --quiet')
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
# ============================================================
# CELL 7 — Split by FLOOD EVENT (prevents leakage)
# ============================================================
def get_event_name(filename):
    return filename.split("_")[0]
 
event_to_chips = {}
for f in s2_files:
    base_name = f.replace("_S2Hand.tif", "")
    label_name = base_name + "_LabelHand.tif"
    if not os.path.exists(os.path.join(label_dir, label_name)):
        continue
    event = get_event_name(f)
    event_to_chips.setdefault(event, []).append(base_name)
 
events = sorted(event_to_chips.keys())
random.Random(SEED).shuffle(events)
n_events = len(events)
n_train = max(1, int(0.7 * n_events))
n_val = max(1, int(0.15 * n_events))
 
train_events = events[:n_train]
val_events = events[n_train:n_train + n_val]
test_events = events[n_train + n_val:]
 
train_chips = [c for e in train_events for c in event_to_chips[e]]
val_chips = [c for e in val_events for c in event_to_chips[e]]
test_chips = [c for e in test_events for c in event_to_chips[e]]
 
print(f"Train events: {train_events} ({len(train_chips)} chips)")
print(f"Val events  : {val_events} ({len(val_chips)} chips)")
print(f"Test events : {test_events} ({len(test_chips)} chips)")
assert set(train_events).isdisjoint(set(val_events))
assert set(train_events).isdisjoint(set(test_events))
assert set(val_events).isdisjoint(set(test_events))
print("PASSED — no event-level leakage.")

Train events: ['Somalia', 'Mekong', 'India', 'Spain', 'Pakistan', 'Paraguay', 'Sri-Lanka'] (291 chips)
Val events  : ['Nigeria'] (18 chips)
Test events : ['Bolivia', 'Ghana', 'USA'] (137 chips)
PASSED — no event-level leakage.


In [9]:
# ============================================================
# CELL 8 — Dataset class
# ============================================================
class FloodDataset(Dataset):
    def __init__(self, chip_names, s2_dir, label_dir, augment=False):
        self.chip_names = chip_names
        self.s2_dir = s2_dir
        self.label_dir = label_dir
        self.augment = augment
 
    def __len__(self):
        return len(self.chip_names)
 
    def __getitem__(self, idx):
        base_name = self.chip_names[idx]
        s2_path = os.path.join(self.s2_dir, base_name + "_S2Hand.tif")
        label_path = os.path.join(self.label_dir, base_name + "_LabelHand.tif")
 
        with rasterio.open(s2_path) as src:
            data = src.read().astype(np.float32)
        with rasterio.open(label_path) as lsrc:
            label = lsrc.read(1).astype(np.float32)
 
        selected = data[[1, 2, 3, 7], :, :]
        for b in range(selected.shape[0]):
            band = selected[b]
            low, high = np.percentile(band, [2, 98])
            if high > low:
                band = np.clip(band, low, high)
                band = (band - low) / (high - low)
            else:
                band = np.zeros_like(band)
            selected[b] = band
 
        valid_mask = (label != -1).astype(np.float32)
        label_clean = np.where(label == -1, 0, label).astype(np.float32)
 
        image_tensor = torch.from_numpy(selected)
        label_tensor = torch.from_numpy(label_clean).unsqueeze(0)
        mask_tensor = torch.from_numpy(valid_mask).unsqueeze(0)
 
        if self.augment:
            if random.random() > 0.5:
                image_tensor = torch.flip(image_tensor, dims=[2])
                label_tensor = torch.flip(label_tensor, dims=[2])
                mask_tensor = torch.flip(mask_tensor, dims=[2])
            if random.random() > 0.5:
                image_tensor = torch.flip(image_tensor, dims=[1])
                label_tensor = torch.flip(label_tensor, dims=[1])
                mask_tensor = torch.flip(mask_tensor, dims=[1])
 
        return image_tensor, label_tensor, mask_tensor
 
 
train_dataset = FloodDataset(train_chips, s2_dir, label_dir, augment=True)
val_dataset = FloodDataset(val_chips, s2_dir, label_dir, augment=False)
test_dataset = FloodDataset(test_chips, s2_dir, label_dir, augment=False)
 
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)
 
sample_img, sample_label, sample_mask = train_dataset[0]
print(f"Image shape: {sample_img.shape} | Label shape: {sample_label.shape} | Mask shape: {sample_mask.shape}")

Image shape: torch.Size([4, 512, 512]) | Label shape: torch.Size([1, 512, 512]) | Mask shape: torch.Size([1, 512, 512])


In [10]:
# ============================================================
# CELL 9 — U-Net model
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)
 
 
class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1, base_filters=32):
        super().__init__()
        f = base_filters
        self.enc1 = DoubleConv(in_channels, f)
        self.enc2 = DoubleConv(f, f * 2)
        self.enc3 = DoubleConv(f * 2, f * 4)
        self.enc4 = DoubleConv(f * 4, f * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(f * 8, f * 16)
        self.up4 = nn.ConvTranspose2d(f * 16, f * 8, 2, stride=2)
        self.dec4 = DoubleConv(f * 16, f * 8)
        self.up3 = nn.ConvTranspose2d(f * 8, f * 4, 2, stride=2)
        self.dec3 = DoubleConv(f * 8, f * 4)
        self.up2 = nn.ConvTranspose2d(f * 4, f * 2, 2, stride=2)
        self.dec2 = DoubleConv(f * 4, f * 2)
        self.up1 = nn.ConvTranspose2d(f * 2, f, 2, stride=2)
        self.dec1 = DoubleConv(f * 2, f)
        self.out_conv = nn.Conv2d(f, out_channels, 1)
 
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)
 
 
model = UNet(in_channels=4, out_channels=1, base_filters=32).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net created. Total parameters: {n_params:,}")
 
with torch.no_grad():
    test_input = torch.randn(2, 4, 512, 512).to(device)
    test_output = model(test_input)
    print(f"Input shape : {test_input.shape}")
    print(f"Output shape: {test_output.shape}")

U-Net created. Total parameters: 7,766,273
Input shape : torch.Size([2, 4, 512, 512])
Output shape: torch.Size([2, 1, 512, 512])


In [11]:
# ============================================================
# CELL 10 — Loss, optimizer, scheduler
# ============================================================
class MaskedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
 
    def forward(self, logits, targets, valid_mask):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        bce = (bce * valid_mask).sum() / (valid_mask.sum() + 1e-6)
        probs = torch.sigmoid(logits)
        probs_masked = probs * valid_mask
        targets_masked = targets * valid_mask
        intersection = (probs_masked * targets_masked).sum()
        union = probs_masked.sum() + targets_masked.sum()
        dice_loss = 1 - (2 * intersection + 1e-6) / (union + 1e-6)
        return self.bce_weight * bce + (1 - self.bce_weight) * dice_loss
 
 
criterion = MaskedBCEDiceLoss(bce_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
print("Loss, optimizer, scheduler ready.")

Loss, optimizer, scheduler ready.


In [12]:
# ============================================================
# CELL 11 — Training loop
# ============================================================
import time
 
def compute_iou_f1(logits, targets, valid_mask, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    valid = valid_mask.bool()
    preds_v = preds[valid]
    targets_v = targets[valid]
    tp = (preds_v * targets_v).sum().item()
    fp = (preds_v * (1 - targets_v)).sum().item()
    fn = ((1 - preds_v) * targets_v).sum().item()
    iou = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    return iou, f1
 
 
N_EPOCHS = 25
best_val_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_f1": []}
 
print(f"Starting training for {N_EPOCHS} epochs on {device}...")
for epoch in range(N_EPOCHS):
    start_time = time.time()
    model.train()
    train_losses = []
    for images, labels, masks in train_loader:
        images, labels, masks = images.to(device), labels.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels, masks)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
 
    model.eval()
    val_losses, val_ious, val_f1s = [], [], []
    with torch.no_grad():
        for images, labels, masks in val_loader:
            images, labels, masks = images.to(device), labels.to(device), masks.to(device)
            logits = model(images)
            loss = criterion(logits, labels, masks)
            val_losses.append(loss.item())
            iou, f1 = compute_iou_f1(logits, labels, masks)
            val_ious.append(iou)
            val_f1s.append(f1)
 
    train_loss, val_loss = np.mean(train_losses), np.mean(val_losses)
    val_iou, val_f1 = np.mean(val_ious), np.mean(val_f1s)
    scheduler.step(val_loss)
 
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    history["val_f1"].append(val_f1)
 
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1:2d}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val IoU: {val_iou:.4f} | Val F1: {val_f1:.4f} | {elapsed:.1f}s")
 
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "/kaggle/working/best_unet_model.pth")
        print(f"  -> New best model saved (Val F1: {val_f1:.4f})")
 
print(f"Training complete. Best validation F1: {best_val_f1:.4f}")
 

Starting training for 25 epochs on cuda...
Epoch  1/25 | Train Loss: 0.6281 | Val Loss: 0.5249 | Val IoU: 0.5143 | Val F1: 0.6183 | 25.3s
  -> New best model saved (Val F1: 0.6183)
Epoch  2/25 | Train Loss: 0.5153 | Val Loss: 0.3721 | Val IoU: 0.5661 | Val F1: 0.6630 | 25.4s
  -> New best model saved (Val F1: 0.6630)
Epoch  3/25 | Train Loss: 0.4610 | Val Loss: 0.3253 | Val IoU: 0.5740 | Val F1: 0.6696 | 26.3s
  -> New best model saved (Val F1: 0.6696)
Epoch  4/25 | Train Loss: 0.4111 | Val Loss: 0.3410 | Val IoU: 0.5142 | Val F1: 0.6282 | 27.6s
Epoch  5/25 | Train Loss: 0.4025 | Val Loss: 0.2961 | Val IoU: 0.5744 | Val F1: 0.6716 | 29.1s
  -> New best model saved (Val F1: 0.6716)
Epoch  6/25 | Train Loss: 0.3798 | Val Loss: 0.3822 | Val IoU: 0.4786 | Val F1: 0.5939 | 28.2s
Epoch  7/25 | Train Loss: 0.3769 | Val Loss: 0.2780 | Val IoU: 0.5783 | Val F1: 0.6752 | 28.1s
  -> New best model saved (Val F1: 0.6752)
Epoch  8/25 | Train Loss: 0.3994 | Val Loss: 0.3876 | Val IoU: 0.4672 | Val F

In [ ]:
# ============================================================
# CELL 12 — SAVE VERSION NOW (manually, in the Kaggle UI)
# before continuing to evaluation.
# ============================================================
 

In [15]:
# CELL 13 — Final TEST evaluation (default threshold 0.5)
# ============================================================
model.load_state_dict(torch.load("/kaggle/working/best_unet_model.pth"))
model.eval()
 
total_tp = total_fp = total_fn = total_correct = total_valid_pixels = 0
with torch.no_grad():
    for images, labels, masks in test_loader:
        images, labels, masks = images.to(device), labels.to(device), masks.to(device)
        logits = model(images)
        preds = (torch.sigmoid(logits) > 0.5).float()
        valid = masks.bool()
        preds_v = preds[valid]
        labels_v = labels[valid]
        total_tp += (preds_v * labels_v).sum().item()
        total_fp += (preds_v * (1 - labels_v)).sum().item()
        total_fn += ((1 - preds_v) * labels_v).sum().item()
        total_correct += (preds_v == labels_v).sum().item()
        total_valid_pixels += valid.sum().item()
 
test_iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
test_precision = total_tp / (total_tp + total_fp + 1e-6)
test_recall = total_tp / (total_tp + total_fn + 1e-6)
test_f1 = 2 * test_precision * test_recall / (test_precision + test_recall + 1e-6)
test_accuracy = total_correct / (total_valid_pixels + 1e-6)
 
print(f"TEST SET RESULTS (n={len(test_chips)} chips, events={test_events})")
print(f"  Accuracy:{test_accuracy:.4f} Precision:{test_precision:.4f} Recall:{test_recall:.4f} "
      f"F1:{test_f1:.4f} IoU:{test_iou:.4f}")

TEST SET RESULTS (n=137 chips, events=['Bolivia', 'Ghana', 'USA'])
  Accuracy:0.9575 Precision:0.6453 Recall:0.6956 F1:0.6695 IoU:0.5032


In [16]:
# ============================================================
# CELL 14 — Per-chip / per-event breakdown (BUG FIXED HERE)
# ============================================================
model.eval()
per_chip_results = []
 
with torch.no_grad():
    for idx in range(len(test_dataset)):
        img, label, mask = test_dataset[idx]
        chip_name = test_chips[idx]
        event = get_event_name(chip_name + "_S2Hand.tif")
 
        img_batch = img.unsqueeze(0).to(device)
        logits = model(img_batch)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().cpu()
 
        # FIX: squeeze() fully (not squeeze(0)) so all three tensors
        # end up [512, 512] before masking — this is the line that
        # caused IndexError before.
        preds_flat = preds.squeeze()
        valid_flat = mask.squeeze()
        label_flat = label.squeeze()
        valid_bool = valid_flat.bool()
 
        preds_v = preds_flat[valid_bool]
        label_v = label_flat[valid_bool]
 
        tp = (preds_v * label_v).sum().item()
        fp = (preds_v * (1 - label_v)).sum().item()
        fn = ((1 - preds_v) * label_v).sum().item()
 
        iou = tp / (tp + fp + fn + 1e-6)
        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        f1 = 2 * precision * recall / (precision + recall + 1e-6)
 
        pred_flood_pct = 100 * preds_v.sum().item() / (preds_v.numel() + 1e-6)
        true_flood_pct = 100 * label_v.sum().item() / (label_v.numel() + 1e-6)
 
        per_chip_results.append({
            "chip": chip_name, "event": event, "iou": iou, "f1": f1,
            "precision": precision, "recall": recall,
            "pred_flood_pct": pred_flood_pct, "true_flood_pct": true_flood_pct
        })
 
print("PER-EVENT TEST BREAKDOWN")
for event in sorted(set(r["event"] for r in per_chip_results)):
    ev = [r for r in per_chip_results if r["event"] == event]
    print(f"{event:10s} | n={len(ev):3d} | IoU: {np.mean([r['iou'] for r in ev]):.4f} | "
          f"F1: {np.mean([r['f1'] for r in ev]):.4f}")

PER-EVENT TEST BREAKDOWN
Bolivia    | n= 15 | IoU: 0.4616 | F1: 0.5594
Ghana      | n= 53 | IoU: 0.2066 | F1: 0.2592
USA        | n= 69 | IoU: 0.3436 | F1: 0.4555


In [17]:
# ============================================================
# CELL 15 — Split flood vs empty chips; fair comparison setup
# ============================================================
flood_chips_results = [r for r in per_chip_results if r["true_flood_pct"] > 0]
empty_chips_results = [r for r in per_chip_results if r["true_flood_pct"] == 0]
 
overall_flood_iou = np.mean([r["iou"] for r in flood_chips_results])
overall_flood_f1 = np.mean([r["f1"] for r in flood_chips_results])
print(f"Flood-containing chips: {len(flood_chips_results)} | Empty chips: {len(empty_chips_results)}")
print(f"U-Net on flood chips only -> IoU: {overall_flood_iou:.4f} | F1: {overall_flood_f1:.4f}")
 
n_clean = sum(1 for r in empty_chips_results if r["pred_flood_pct"] == 0)
print(f"Empty chips predicted with ZERO false flood: {n_clean}/{len(empty_chips_results)}")

Flood-containing chips: 111 | Empty chips: 26
U-Net on flood chips only -> IoU: 0.3746 | F1: 0.4825
Empty chips predicted with ZERO false flood: 6/26


In [18]:
# ============================================================
# CELL 16 — Fair NDWI baseline on SAME test chips
# ============================================================
ndwi_test_results = []
for base_name in test_chips:
    s2_path = os.path.join(s2_dir, base_name + "_S2Hand.tif")
    label_path = os.path.join(label_dir, base_name + "_LabelHand.tif")
    with rasterio.open(s2_path) as src:
        data = src.read().astype(np.float32)
    with rasterio.open(label_path) as lsrc:
        label = lsrc.read(1)
    green, nir = data[2], data[7]
    denom = green + nir
    denom[denom == 0] = 1e-6
    ndwi = (green - nir) / denom
    pred = (ndwi > NDWI_BEST_THRESHOLD).astype(np.int8)
    valid_mask = (label != -1)
    y_true = label[valid_mask].astype(np.int8)
    y_pred = pred[valid_mask]
    event = get_event_name(base_name + "_S2Hand.tif")
    true_flood_pct = 100 * y_true.sum() / len(y_true) if len(y_true) > 0 else 0
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    iou = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    ndwi_test_results.append({"chip": base_name, "event": event, "iou": iou, "f1": f1, "true_flood_pct": true_flood_pct})
 
ndwi_flood = [r for r in ndwi_test_results if r["true_flood_pct"] > 0]
ndwi_overall_iou = np.mean([r["iou"] for r in ndwi_flood])
ndwi_overall_f1 = np.mean([r["f1"] for r in ndwi_flood])
 
print("HEAD-TO-HEAD ON FLOOD-CONTAINING TEST CHIPS (fair comparison)")
print(f"  NDWI  -> IoU: {ndwi_overall_iou:.4f} | F1: {ndwi_overall_f1:.4f}")
print(f"  U-Net -> IoU: {overall_flood_iou:.4f} | F1: {overall_flood_f1:.4f}")
print(f"  Delta -> IoU: {overall_flood_iou - ndwi_overall_iou:+.4f} | F1: {overall_flood_f1 - ndwi_overall_f1:+.4f}")

HEAD-TO-HEAD ON FLOOD-CONTAINING TEST CHIPS (fair comparison)
  NDWI  -> IoU: 0.5540 | F1: 0.6614
  U-Net -> IoU: 0.3746 | F1: 0.4825
  Delta -> IoU: -0.1793 | F1: -0.1789
